# Data Quality Checker - Complete Example

This notebook demonstrates comprehensive data quality checking capabilities including:
- ✅ Table-level checks (nulls, ranges, patterns, uniqueness, etc.)
- ✅ Cross-table checks (foreign keys, referential integrity)
- ✅ AI-powered semantic validation
- ✅ Schema validation
- ✅ Volume checks

## Architecture

```
Data Generator → CSV/JSON → Load to Tables → Data Quality Checks → Results & Reports
```

## Step 1: Generate Sample Data

First, let's generate some test data using the data generator.

In [ ]:
import sys
sys.path.append("/Workspace/oracle-aidp-samples/data_generator")
sys.path.append("/Workspace/oracle-aidp-samples/data_quality")

from data_generator import MultiTableDataGenerator

# Initialize generator
gen = MultiTableDataGenerator(seed=42)

# Configuration for sample data
config = {
    'output_path': '/Workspace/dq_demo_data',
    'output_format': 'csv',
    'tables': [
        {
            'table_name': 'users',
            'rows_count': 100,
            'columns': [
                {'name': 'user_id', 'type': 'integer', 'range': [1, 100], 'unique': True},
                {'name': 'name', 'type': 'string', 'length': 10, 'prefix': 'User_'},
                {'name': 'email', 'type': 'email', 'domains': ['example.com', 'test.com'], 'unique': True},
                {'name': 'age', 'type': 'integer', 'range': [18, 80]},
                {'name': 'created_date', 'type': 'date', 'format': '%Y-%m-%d'}
            ]
        },
        {
            'table_name': 'orders',
            'rows_count': 500,
            'columns': [
                {'name': 'order_id', 'type': 'integer', 'range': [1, 500], 'unique': True},
                {'name': 'user_id', 'type': 'reference', 'ref_table': 'users', 'ref_column': 'user_id'},
                {'name': 'product', 'type': 'choice', 'values': ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard']},
                {'name': 'amount', 'type': 'float', 'range': [10.0, 5000.0], 'decimals': 2},
                {'name': 'status', 'type': 'choice', 'values': ['pending', 'completed', 'cancelled', 'shipped']},
                {'name': 'order_date', 'type': 'date', 'format': '%Y-%m-%d'}
            ]
        }
    ]
}

# Generate data
results = gen.generate_from_config(config)

# Print summary
for table_name, result in results.items():
    print(f"✓ Generated {result['rows_count']} rows for {table_name}")

## Step 2: Load Data into Spark Tables

In [ ]:
from pyspark.sql import SparkSession

# Get Spark session
spark = SparkSession.builder.appName("DQ_Demo").getOrCreate()

# Load users table
users_df = spark.read.csv(
    "/Workspace/dq_demo_data/users.csv",
    header=True,
    inferSchema=True
)

# Load orders table
orders_df = spark.read.csv(
    "/Workspace/dq_demo_data/orders.csv",
    header=True,
    inferSchema=True
)

# Create temporary tables
users_df.createOrReplaceTempView("users")
orders_df.createOrReplaceTempView("orders")

# Or save as managed tables
users_df.write.mode("overwrite").saveAsTable("default.users")
orders_df.write.mode("overwrite").saveAsTable("default.orders")

print("✓ Tables loaded successfully")
print(f"  Users: {users_df.count()} rows")
print(f"  Orders: {orders_df.count()} rows")

## Step 3: Preview the Data

In [ ]:
print("Users Table Schema:")
users_df.printSchema()

print("\nUsers Sample Data:")
users_df.show(5, truncate=False)

print("\nOrders Sample Data:")
orders_df.show(5, truncate=False)

## Step 4: Run Data Quality Checks

Now let's run comprehensive data quality checks using the configuration.

In [ ]:
from jobs.run_dq_job import run_dq_checks

# Run DQ checks with demo configuration
config_path = "/Workspace/oracle-aidp-samples/data_quality/config/demo_dq_config.yaml"

results_df = run_dq_checks(config_path, spark)

## Step 5: Analyze Results

In [ ]:
# Show all results
if results_df:
    print("\n" + "="*100)
    print("ALL DATA QUALITY CHECK RESULTS")
    print("="*100)
    results_df.show(truncate=False, n=1000)

In [ ]:
# Filter failed checks
if results_df:
    failed_checks = results_df.filter(results_df.status == "FAIL")
    
    if failed_checks.count() > 0:
        print("\n❌ FAILED CHECKS:")
        failed_checks.show(truncate=False)
    else:
        print("\n🎉 All checks passed!")

In [ ]:
# Analyze by severity
if results_df:
    print("\n📊 Results by Severity:")
    results_df.groupBy("severity", "status").count().orderBy("severity", "status").show()

In [ ]:
# Analyze by table
if results_df:
    print("\n📋 Results by Table:")
    results_df.groupBy("table", "status").count().orderBy("table", "status").show()

## Step 6: Individual Rule Testing

You can also test individual rules programmatically.

In [ ]:
from dq.native_rules import run_not_null, run_unique, run_range, run_pattern

# Test individual rules
print("Testing individual rules on users table:")

# Test NOT NULL
result = run_not_null(users_df, {"column": "user_id"})
print(f"  user_id NOT NULL: {'✓ PASS' if result else '✗ FAIL'}")

# Test UNIQUE
result = run_unique(users_df, {"column": "email"})
print(f"  email UNIQUE: {'✓ PASS' if result else '✗ FAIL'}")

# Test RANGE
result = run_range(users_df, {"column": "age", "min": 18, "max": 100})
print(f"  age RANGE (18-100): {'✓ PASS' if result else '✗ FAIL'}")

# Test PATTERN
result = run_pattern(users_df, {
    "column": "email",
    "pattern": r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
})
print(f"  email PATTERN: {'✓ PASS' if result else '✗ FAIL'}")

## Step 7: Test Cross-Table Rules

In [ ]:
from dq.cross_table_rules import run_foreign_key_rule

# Test foreign key relationship
fk_rule = {
    "source": {"column": "user_id"},
    "target": {"column": "user_id"},
    "allow_nulls": False
}

result = run_foreign_key_rule(orders_df, users_df, fk_rule)
print(f"\nForeign Key Check (orders.user_id → users.user_id): {'✓ PASS' if result else '✗ FAIL'}")

## Step 8: AI-Powered Semantic Validation (Optional)

If you want to use AI-powered validation, ensure you have access to OCI GenAI models.

In [ ]:
from dq.llm_rules import run_llm_rule

# Example LLM rule (requires OCI GenAI access)
llm_rule = {
    "column": "email",
    "description": "valid professional email addresses",
    "model": "meta.llama-3.1-70b-instruct",
    "sample_size": 20
}

# Uncomment to run (requires GenAI access)
# result = run_llm_rule(users_df, llm_rule, spark)
# print(f"\nLLM Semantic Check: {result}")

print("\n⚠️  LLM checks require OCI GenAI access. Configure model access in AIDP.")

## Step 9: Save Results for Reporting

In [ ]:
if results_df:
    # Save results to a table
    results_df.write.mode("overwrite").saveAsTable("default.dq_results")
    print("✓ Results saved to table: default.dq_results")
    
    # Or save as CSV
    results_df.coalesce(1).write.mode("overwrite").csv(
        "/Workspace/dq_results",
        header=True
    )
    print("✓ Results saved to CSV: /Workspace/dq_results")

## Step 10: Create Custom Configuration

You can create your own DQ configuration for your specific tables.

In [ ]:
custom_config = """
tables:
  my_table:
    table_name: default.my_table
    
    rules:
      - name: id_not_null
        type: not_null
        column: id
        severity: CRITICAL
        weight: 10
      
      - name: id_unique
        type: unique
        column: id
        severity: CRITICAL
        weight: 10
      
      - name: amount_range
        type: range
        column: amount
        min: 0
        max: 100000
        severity: HIGH
        weight: 7
"""

# Save custom config
with open("/Workspace/my_dq_config.yaml", "w") as f:
    f.write(custom_config)

print("✓ Custom configuration created: /Workspace/my_dq_config.yaml")

## Summary

This notebook demonstrated:

1. ✅ Generating sample data with the data generator
2. ✅ Loading data into Spark tables
3. ✅ Running comprehensive data quality checks
4. ✅ Analyzing results and identifying issues
5. ✅ Testing individual and cross-table rules
6. ✅ AI-powered semantic validation
7. ✅ Saving results for reporting

### Available Check Types:

- **not_null**: Check for null values
- **unique**: Check for uniqueness
- **range**: Validate numeric ranges
- **pattern**: Regex pattern matching
- **duplicate**: Find duplicate rows
- **completeness**: Validate data completeness percentage
- **type_check**: Validate data types
- **value_set**: Check against allowed values
- **string_length**: Validate string length
- **foreign_key**: Cross-table foreign key validation
- **llm_semantic**: AI-powered semantic validation

### Next Steps:

1. Customize the configuration for your tables
2. Schedule regular DQ checks as jobs
3. Set up alerts for failed checks
4. Integrate with your data pipeline